# DrugReview: readouts, attention factors, evidence, and localization

This notebook reproduces the manuscript's results on what the aspect information predicts, where the attention factors settle, and how well the evidence design finds the annotator's passages. It trains no encoder: it reads the saved predictions, factor loadings, evidence metrics, and tokenizers of the rung notebooks (`03`, `05`, `06`, `07`), and fits only the stated readouts of the fixed rung-3 predictions. Every result is printed below and written to `08_Manuscript_evaluation/outputs/`.

| Section | Result | Manuscript |
|---|---|---|
| 1 | Fixed readouts of the rung-3 aspect probabilities and the no-text lookup rules | Table S11, Section 4.2.2 |
| 2 | Rung 6 factors: mean fusion weight, attention lift on the passages, most-attended tokens | Table S13, Section 4.2.2 |
| 3 | Rung 7 evidence head | Table 11 |
| 4 | Localization gain from evidence supervision | Table 12 |
| 5 | Label-free localization baselines | Table S14, Section 4.2.3 |

Intervals are 95% percentile intervals from a bootstrap of narrative groups on the test partition, 1,000 draws with seed 2025; the paired differences use the same draws for both members. Section 5 needs the Hugging Face `transformers` library and the fast tokenizers saved with the rung-7 runs.

## 0. Setup: paths, constants, bootstrap

In [1]:
import os, json, glob, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, cohen_kappa_score
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

# ROOT is the folder that holds 00_labelling/, 00_split/, 01_OriginalLabel/, ... (Drive: NLP_Projects/03_DrugReview; local: "01. Colab backup").
if os.path.isdir("/content") and not os.environ.get("DRUGREVIEW_ROOT"):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)

def resolve_project_root():
    override = os.environ.get("DRUGREVIEW_ROOT")
    if override:
        candidate = Path(override).expanduser().resolve()
        if not candidate.is_dir():
            raise FileNotFoundError(f"DRUGREVIEW_ROOT does not exist: {candidate}")
        return candidate
    for base in [Path.cwd(), *Path.cwd().parents, Path("/content/drive/MyDrive/NLP_Projects/03_DrugReview")]:
        for candidate in (base, base / "DrugReview_new"):
            if (candidate / "01_OriginalLabel").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set DRUGREVIEW_ROOT to the folder containing 01_OriginalLabel, 00_split, and the run data.")

ROOT = resolve_project_root()
OUT = Path(os.environ.get("DRUGREVIEW_EVAL_OUT", ROOT / "08_Manuscript_evaluation" / "outputs")); OUT.mkdir(parents=True, exist_ok=True)

PROTOCOL_ID = "v2_ef356fdf87"
REL = ROOT / "00_labelling" / "output_v2" / PROTOCOL_ID
CLASSES = ["VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "VERY_POSITIVE"]
A3 = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
ASPECTS = ["efficacy", "safety", "burden", "cost"]
ENCODERS = ("albert", "biobert")
RUNS = {"03": "03_Full_aspects/outputs/aspect_auxiliary_v2_3class",
        "04": "04_Holistic_all_aspects/outputs/aspect_auxiliary_v2_3class",
        "A0": "ablation/A0_Matched_holistic_control/outputs/aspect_auxiliary_v2_3class",
        "5p": "05_Confirmatory/outputs/multilevel_representation_only_v2",
        "5r": "05_Confirmatory/outputs/multilevel_representation_residual_v2",
        "5j": "05_Confirmatory/outputs/multilevel_probability_bottleneck_v2",
        "5d": "05_Confirmatory/outputs/multilevel_detached_probability_bottleneck_v2",
        "6ff": "06_Semi_confirmatory/outputs/semi_4anchored_4free_v2",
        "6a": "06_Semi_confirmatory/outputs/semi_4anchored_0free_v2",
        "6f": "06_Semi_confirmatory/outputs/semi_0anchored_4free_v2",
        "7e": "07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_1",
        "7c": "07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_0"}
NAMES = {"03": "3 aspect baseline", "04": "4 parallel", "A0": "4 no aspect heads", "5p": "5 fusion", "5r": "5 residual",
         "5j": "5 bottleneck, joint", "5d": "5 bottleneck, detached", "6ff": "6 4 anchored + 4 free", "6a": "6 4 anchored + 0 free",
         "6f": "6 0 anchored + 4 free", "7e": "7 evidence loss on", "7c": "7 evidence loss off"}

def test_predictions(run, enc, usecols=None):
    df = pd.read_csv(ROOT / RUNS[run] / enc / f"{enc}_test_predictions.csv", usecols=usecols)
    return df.sort_values("uniqueID").reset_index(drop=True)

# Narrative-group bootstrap on group-level confusion matrices: each draw resamples the narrative groups of the test partition
# with replacement (numpy default_rng(seed).integers(0, G, G)), so reviews that share a narrative stay together, and every
# model scored on the same draws is paired. 95% percentile intervals.
def confusion_metrics(M):
    """M: (..., K, K) counts, rows = true. Returns weighted F1, macro F1, quadratic-weighted kappa."""
    K = M.shape[-1]; W = np.array([[(i - j) ** 2 for j in range(K)] for i in range(K)], float)
    tp = np.einsum("...ii->...i", M); rows = M.sum(-1); cols = M.sum(-2); n = rows.sum(-1, keepdims=True)
    den = 2 * tp + (cols - tp) + (rows - tp); f1 = np.where(den > 0, 2 * tp / np.where(den > 0, den, 1), 0.0)
    wf1 = (f1 * rows).sum(-1) / n[..., 0]; mf1 = f1.mean(-1)
    E = rows[..., :, None] * cols[..., None, :] / n[..., None]; qwk = 1 - (W * M).sum((-1, -2)) / (W * E).sum((-1, -2))
    return wf1, mf1, qwk

def group_draws(groups, n, seed):
    codes = pd.factorize(pd.Series(groups))[0]; G = int(codes.max() + 1); rng = np.random.default_rng(seed)
    counts = np.empty((n, G))                              # filled in place: 10,000 x ~19,000 draws stay near 1.5 GB
    for i in range(n):
        counts[i] = np.bincount(rng.integers(0, G, G), minlength=G)
    return codes, G, counts

def boot_metrics(y, preds, groups, K, n=1000, seed=2025):
    """preds: dict name -> predicted ids (aligned with y). Returns dict name -> (point (wf1, mf1, qwk), draws (n, 3))."""
    codes, G, cnt = group_draws(groups, n, seed); y = np.asarray(y); out = {}
    for k, p in preds.items():
        C = np.zeros((G, K * K)); np.add.at(C, (codes, y * K + np.asarray(p)), 1.0)
        out[k] = (np.array(confusion_metrics(C.sum(0).reshape(K, K))), np.stack(confusion_metrics((cnt @ C).reshape(n, K, K)), 1))
    return out

def ci(a, axis=0):
    return np.percentile(a, [2.5, 97.5], axis=axis)

def fmt_ci(x, lo, hi, d=3):
    return f"{x:.{d}f} ({lo:.{d}f}, {hi:.{d}f})"

def save(df, name, **kw):
    df.to_csv(OUT / name, **kw); print(f"written: outputs/{name}")

print("ROOT:", ROOT); print("OUT :", OUT)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/NLP_Projects/03_DrugReview
OUT : /content/drive/MyDrive/NLP_Projects/03_DrugReview/08_Manuscript_evaluation/outputs


## 1. Fixed readouts and the no-text lookup rules

These references hold the aspect information fixed. The lookup rules use only the annotated aspect labels; the readouts use only the twelve class probabilities of the rung-3 aspect model, chosen by five-fold narrative-group cross-validation on the validation partition (the groups are dealt to the folds by a seeded permutation, so the folds are the same on any machine) and scored once on the test partition. Rung 5's detached bottleneck and fusion configuration are scored on the same draws for comparison.

In [2]:
# Fixed readouts of the rung-3 aspect probabilities: each readout is chosen by 5-fold narrative-group cross-validation on the validation
# partition, refitted there, and scored once on the test partition, so the aspect predictor itself is never refitted. The no-text
# lookup rules assign each combination of the annotated aspect labels (three classes, or the five source states) its most frequent
# holistic label among training reviews; unseen combinations receive the training majority class.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")
SEED = 2025
rel = pd.read_csv(REL / f"drugscom_plus_ai_labels_{PROTOCOL_ID}.csv", usecols=["uniqueID", "eligible_both_streams", "ai_sent5_hard"] + [f"ai_{a}" for a in ASPECTS] + [f"ai_{a}_state" for a in ASPECTS])
split = pd.read_csv(ROOT / "00_split" / "data" / "DrugReview_split.csv")
d = rel[rel.eligible_both_streams].merge(split, on="uniqueID", validate="one_to_one"); d["y"] = d.ai_sent5_hard.map({c: i for i, c in enumerate(CLASSES)}).astype(int)
tr, te = d[d.split == "train"], d[d.split == "test"].sort_values("uniqueID").reset_index(drop=True)
def lookup(cols):
    ktr = tr[cols].astype(str).agg("|".join, axis=1); kte = te[cols].astype(str).agg("|".join, axis=1)
    table = tr.groupby(ktr.values).y.agg(lambda s: s.value_counts().idxmax()); maj = int(tr.y.value_counts().idxmax())
    return kte.map(table).fillna(maj).astype(int).values, len(table), float(kte.isin(table.index).mean())
p3, n3, cov3 = lookup([f"ai_{a}" for a in ASPECTS]); p5, n5, cov5 = lookup([f"ai_{a}_state" for a in ASPECTS])
print(f"lookup rules: three-class labels {n3} combinations seen in training (test coverage {cov3:.4f}); five source states {n5} (coverage {cov5:.4f})")
rows = []
for enc in ENCODERS:
    v = pd.read_csv(ROOT / RUNS["03"] / enc / f"{enc}_val_predictions.csv"); t = test_predictions("03", enc)
    assert (t.uniqueID.values == te.uniqueID.values).all()
    for a in ASPECTS: assert (t[f"{a}_true_id"].values == te[f"ai_{a}"].map({c: i for i, c in enumerate(A3)}).values).all(), a
    feats = [f"{a}_p_{c}" for a in ASPECTS for c in A3]; v = v.merge(d[["uniqueID", "y", "narrative_group"]].rename(columns={"narrative_group": "ng"}), on="uniqueID", validate="one_to_one")
    Xv, yv, gv = v[feats].values, v.y.values, v.ng.values; Xt, yt = t[feats].values, te.y.values
    # Five folds of narrative groups: the groups, in sorted order, are permuted with the seed and dealt to the folds in turn, so the
    # folds are the same on any machine (scikit-learn's GroupKFold breaks ties between equal-sized groups by a platform-dependent sort).
    ug = np.unique(gv); fold_of_group = np.empty(len(ug), int); fold_of_group[np.random.default_rng(SEED).permutation(len(ug))] = np.arange(len(ug)) % 5
    fold = fold_of_group[np.searchsorted(ug, gv)]
    cands = {"LR, probabilities": [LogisticRegression(C=c, max_iter=5000) for c in (0.1, 1, 10, 100)],
             "LR, log-probabilities": [LogisticRegression(C=c, max_iter=5000) for c in (0.1, 1, 10, 100)],
             "LR, probabilities, balanced": [LogisticRegression(C=c, max_iter=5000, class_weight="balanced") for c in (0.1, 1, 10, 100)],
             "Gradient boosting": [HistGradientBoostingClassifier(learning_rate=lr, max_leaf_nodes=ml, max_iter=300, random_state=SEED) for lr in (0.03, 0.1) for ml in (15, 31)],
             "MLP": [MLPClassifier(hidden_layer_sizes=h, max_iter=500, early_stopping=True, random_state=SEED) for h in ((64,), (128, 64))]}
    preds, chosen = {}, {}
    for name, models in cands.items():
        X_v = np.log(np.clip(Xv, 1e-6, 1)) if "log" in name else Xv; X_t = np.log(np.clip(Xt, 1e-6, 1)) if "log" in name else Xt; best = None
        for m in models:
            sc = []
            for k in range(5):
                i_tr, i_va = np.where(fold != k)[0], np.where(fold == k)[0]
                m.fit(X_v[i_tr], yv[i_tr]); sc.append(f1_score(yv[i_va], m.predict(X_v[i_va]), average="weighted"))
            if best is None or np.mean(sc) > best[0]: best = (np.mean(sc), m)
        m = best[1]; m.fit(X_v, yv); preds[name] = m.predict(X_t); gp = m.get_params()
        chosen[name] = f"{best[0]:.4f} cv wF1; " + ("C=%s" % gp["C"] if "C" in gp else "lr=%s, leaves=%s" % (gp["learning_rate"], gp["max_leaf_nodes"]) if "max_leaf_nodes" in gp else "hidden=%s" % (gp["hidden_layer_sizes"],))
    p5d = test_predictions("5d", enc, ["uniqueID", "overall_true_id", "overall_pred_id"]); p5p = test_predictions("5p", enc, ["uniqueID", "overall_true_id", "overall_pred_id"])
    assert (p5d.uniqueID.values == te.uniqueID.values).all() and (p5d.overall_true_id.values == yt).all() and (p5p.overall_true_id.values == yt).all()
    preds.update({"Lookup, 3-class labels": p3, "Lookup, 5 source states": p5, "5 detached bottleneck": p5d.overall_pred_id.values, "5 fusion": p5p.overall_pred_id.values})
    bs = boot_metrics(yt, preds, te.narrative_group.values, 5, 1000, SEED)
    for name, (pt, bt) in bs.items():
        lo, hi = ci(bt); rows.append({"encoder": enc, "readout": name, "selection": chosen.get(name, ""), "wf1": fmt_ci(pt[0], lo[0], hi[0]), "mf1": fmt_ci(pt[1], lo[1], hi[1]), "qwk": fmt_ci(pt[2], lo[2], hi[2])})
    for a, b in [("Gradient boosting", "LR, probabilities"), ("MLP", "LR, probabilities"), ("LR, probabilities", "5 detached bottleneck"), ("LR, log-probabilities", "5 detached bottleneck"),
                 ("Gradient boosting", "5 detached bottleneck"), ("Gradient boosting", "5 fusion"), ("Lookup, 5 source states", "Lookup, 3-class labels"), ("LR, log-probabilities", "LR, probabilities")]:
        dpt = bs[a][0] - bs[b][0]; lo, hi = ci(bs[a][1] - bs[b][1])
        rows.append({"encoder": enc, "readout": f"{a} - {b}", "selection": "paired difference", "wf1": fmt_ci(dpt[0], lo[0], hi[0]), "mf1": fmt_ci(dpt[1], lo[1], hi[1]), "qwk": fmt_ci(dpt[2], lo[2], hi[2])})
s12 = pd.DataFrame(rows); print(s12.to_string(index=False)); save(s12, "fixed_readouts.csv", index=False)

lookup rules: three-class labels 81 combinations seen in training (test coverage 1.0000); five source states 368 (coverage 0.9986)
encoder                                          readout                         selection                     wf1                     mf1                     qwk
 albert                                LR, probabilities              0.6012 cv wF1; C=100    0.600 (0.593, 0.607)    0.483 (0.477, 0.488)    0.806 (0.800, 0.812)
 albert                            LR, log-probabilities                0.6312 cv wF1; C=1    0.635 (0.628, 0.642)    0.514 (0.509, 0.520)    0.836 (0.830, 0.842)
 albert                      LR, probabilities, balanced              0.5876 cv wF1; C=100    0.582 (0.575, 0.589)    0.499 (0.492, 0.505)    0.813 (0.807, 0.819)
 albert                                Gradient boosting 0.6314 cv wF1; lr=0.03, leaves=31    0.633 (0.625, 0.639)    0.517 (0.510, 0.524)    0.831 (0.824, 0.836)
 albert                                              M

## 2. Rung 6: where the factors settle

The lift divides a factor's attention mass on the tokens of an aspect's passages by the mass that uniform attention over the eligible tokens would place there; 1 is chance. The token shares sum each token's attention over the free factor's listed top tokens across the audit reviews.

In [3]:
# Rung 6: each factor's mean fusion weight on the test partition, its attention lift on the annotator's passages for each aspect
# (attention mass on passage tokens over reviews with a visible passage, divided by the mass uniform attention would place there),
# and, for the free factors, the tokens that carry most of their top attention over the audit reviews.
import collections
rows = []
for run in ("6a", "6ff", "6f"):
    for enc in ENCODERS:
        t = test_predictions(run, enc); fl = pd.read_csv(ROOT / RUNS[run] / enc / f"{enc}_test_factor_loadings.csv")
        for factor in list(dict.fromkeys(fl.factor)):
            beta = t[f"fusion_w_{factor}"].mean(); lift = fl[fl.factor == factor].set_index("aspect").lift
            rows.append({"design": NAMES[run], "encoder": enc, "factor": factor, "mean_beta": round(beta, 3), **{f"lift_{a}": round(lift[a], 2) for a in ASPECTS}})
s14 = pd.DataFrame(rows); print(s14.to_string(index=False)); save(s14, "factor_loadings.csv", index=False)
rows = []
for run in ("6ff", "6f"):
    for enc in ENCODERS:
        a = pd.read_csv(ROOT / RUNS[run] / enc / f"{enc}_audit_predictions.csv")
        for f in range(1, 5):
            tot = collections.Counter()
            for s in a[f"free_{f}_top_tokens"].dropna():
                for item in s.split("|"):
                    tok, w = item.rsplit(":", 1); tot[tok.replace("▁", "").replace("##", "").lower()] += float(w)
            T = sum(tot.values())
            rows.append({"design": NAMES[run], "encoder": enc, "factor": f"free_{f}", "audit_reviews": int(a[f"free_{f}_top_tokens"].notna().sum()),
                         "top_tokens_share_of_top_attention": ", ".join(f"{k} {v / T:.2f}" for k, v in tot.most_common(6))})
tok = pd.DataFrame(rows); print("\n" + tok.to_string(index=False)); save(tok, "free_factor_tokens.csv", index=False)

               design encoder   factor  mean_beta  lift_efficacy  lift_safety  lift_burden  lift_cost
6 4 anchored + 0 free  albert efficacy      0.404           1.40         0.71         0.71       0.38
6 4 anchored + 0 free  albert   safety      0.157           0.78         2.97         0.98       0.52
6 4 anchored + 0 free  albert   burden      0.140           0.80         1.04         2.00       1.93
6 4 anchored + 0 free  albert     cost      0.298           1.00         0.89         1.05       4.25
6 4 anchored + 0 free biobert efficacy      0.295           1.19         0.74         0.74       0.38
6 4 anchored + 0 free biobert   safety      0.246           0.89         2.03         0.93       0.44
6 4 anchored + 0 free biobert   burden      0.219           0.85         0.92         1.73       1.29
6 4 anchored + 0 free biobert     cost      0.241           0.98         0.98         1.07       4.40
6 4 anchored + 4 free  albert efficacy      0.047           1.37         0.55     

## 3. Rung 7: the evidence head

Recorded by the evidence notebooks on the test partition; the $\eta=0$ run has no evidence target, so only its localization is scored.

In [4]:
# Rung 7, the evidence head on the test partition, as recorded by the evidence notebooks: known sentence targets, three-class macro F1,
# F1 for any evidence, direction accuracy on passage sentences, and localization (share of reviews with a visible passage whose
# most-attended sentence contains one, and mean attention mass on passage sentences). The classification scores are recomputed
# here from the saved evidence confusion matrices as a check.
rows = []
for eta, run in ((1, "7e"), (0, "7c")):
    for enc in ENCODERS:
        m = pd.read_csv(ROOT / RUNS[run] / enc / "metrics.csv"); m = m[(m.partition == "test") & m.task.str.startswith("evidence_")]
        for a in ASPECTS:
            r = m[m.task == f"evidence_{a}"].iloc[0]
            row = {"aspect": a, "eta": eta, "encoder": enc, "known_evidence": int(r.known_positive_n), "known_none": int(r.known_negative_n),
                   "macro_f1": round(r.macro_f1, 3) if eta == 1 else "---", "mention_f1": round(r.known_target_mention_f1, 3) if eta == 1 else "---",
                   "direction_accuracy": round(r.selected_positive_direction_accuracy, 3) if eta == 1 else "---",
                   "localization_reviews": int(r.localization_reviews_n), "top1_hit": round(r.top1_selected_quote_hit, 3), "passage_mass": round(r.selected_quote_attention_mass, 3)}
            if eta == 1:
                C = pd.read_csv(ROOT / RUNS[run] / enc / f"{enc}_evidence_{a}_confusion.csv", index_col=0).loc[["NONE", "FAVOURABLE", "UNFAVOURABLE"], ["NONE", "FAVOURABLE", "UNFAVOURABLE"]].values.astype(float)
                tp = np.diag(C); f1 = 2 * tp / (C.sum(0) + C.sum(1)); ev = C[1:].sum(); hit = C[1:, 1:].sum()
                check = (f1.mean(), 2 * hit / (2 * hit + C[1:, 0].sum() + C[0, 1:].sum()), (C[1, 1] + C[2, 2]) / ev)
                assert np.allclose(check, (r.macro_f1, r.known_target_mention_f1, r.selected_positive_direction_accuracy), atol=5e-4), (enc, a, check)
            rows.append(row)
order = {a: i for i, a in enumerate(ASPECTS)}
t10 = pd.DataFrame(rows).sort_values(["aspect", "eta", "encoder"], key=lambda s: s.map(order) if s.name == "aspect" else (-s if s.name == "eta" else s))
print(t10.to_string(index=False)); save(t10, "evidence_head.csv", index=False)

  aspect  eta encoder  known_evidence  known_none macro_f1 mention_f1 direction_accuracy  localization_reviews  top1_hit  passage_mass
efficacy    1  albert           36774        1017    0.833      0.991              0.933                 19209     0.710         0.314
efficacy    1 biobert           36772        1017    0.848      0.992              0.932                 19208     0.665         0.309
efficacy    0  albert           36774        1017      ---        ---                ---                 19209     0.648         0.415
efficacy    0 biobert           36772        1017      ---        ---                ---                 19208     0.600         0.337
  safety    1  albert           22780       35193    0.879      0.921               0.89                 14333     0.816         0.359
  safety    1 biobert           22780       35193    0.867      0.916              0.895                 14333     0.793         0.318
  safety    0  albert           22780       35193      

## 4. Localization gain from evidence supervision

In [5]:
# Localization gain from evidence supervision: among test reviews with a visible passage for the aspect, the share whose most-attended
# sentence contains one of its passages, for the run without the evidence loss (eta=0) and with it (eta=1); paired narrative-group
# bootstrap of the difference, 1,000 draws, seed 2025, in the review order of the saved eta=1 predictions.
rows=[]
for enc in ("albert","biobert"):
    e1=pd.read_csv(ROOT / RUNS["7e"]/enc/f"{enc}_test_predictions.csv"); e0=pd.read_csv(ROOT / RUNS["7c"]/enc/f"{enc}_test_predictions.csv")
    m=e1.merge(e0,on="uniqueID",suffixes=("_1","_0"),validate="one_to_one"); assert (m.narrative_group_1==m.narrative_group_0).all()
    codes=pd.factorize(m.narrative_group_1)[0]; members=[np.flatnonzero(codes==g) for g in range(codes.max()+1)]
    rng=np.random.default_rng(2025); draws=[np.concatenate([members[g] for g in rng.integers(0,len(members),len(members))]) for _ in range(1000)]
    tomap={True:1.0,False:0.0,"True":1.0,"False":0.0}
    for a in ASPECTS:
        vis=m[f"{a}_has_visible_selected_quote_1"].astype(bool).values; assert (vis==m[f"{a}_has_visible_selected_quote_0"].astype(bool).values).all()
        h1=m[f"{a}_top1_selected_quote_hit_1"].map(tomap).values; h0=m[f"{a}_top1_selected_quote_hit_0"].map(tomap).values
        point=np.nanmean(h1[vis])-np.nanmean(h0[vis]); boot=[np.nanmean(h1[i[vis[i]]])-np.nanmean(h0[i[vis[i]]]) for i in draws]; lo,hi=np.percentile(boot,[2.5,97.5])
        rows.append({"encoder":enc,"aspect":a,"n_reviews_with_quote":int(vis.sum()),"hit_eta0":round(float(np.nanmean(h0[vis])),3),"hit_eta1":round(float(np.nanmean(h1[vis])),3),"delta":round(point,3),"ci_lo":round(lo,3),"ci_hi":round(hi,3)})
t11 = pd.DataFrame(rows); print(t11.to_string(index=False)); save(t11, "localization_gain.csv", index=False)

encoder   aspect  n_reviews_with_quote  hit_eta0  hit_eta1  delta  ci_lo  ci_hi
 albert efficacy                 19209     0.648     0.710  0.063  0.055  0.071
 albert   safety                 14333     0.788     0.816  0.028  0.022  0.035
 albert   burden                 10813     0.604     0.657  0.053  0.044  0.061
 albert     cost                  1423     0.807     0.888  0.081  0.058  0.103
biobert efficacy                 19208     0.600     0.665  0.065  0.056  0.073
biobert   safety                 14333     0.711     0.793  0.081  0.073  0.089
biobert   burden                 10812     0.584     0.646  0.062  0.053  0.072
biobert     cost                  1423     0.724     0.881  0.157  0.131  0.180
written: outputs/localization_gain.csv


## 5. Label-free localization baselines

The visible sentences and passages are re-derived with each run's saved tokenizer and checked against the eligibility flag the run exported. Reviews with a single candidate sentence are hits for every chooser; they appear only in the rows for one candidate and for all reviews.

In [6]:
# Label-free localization baselines. The visible sentences and the visible-passage mask of the evidence notebook are re-derived for the
# test partition with each run's saved fast tokenizer (functions copied unchanged from notebook 07), the eligibility flag is checked
# against the run's exported column, and three choosers that use no label or attention (a uniformly random candidate sentence, the
# first sentence, the longest visible sentence) are scored beside the eta=1 and eta=0 models, by number of candidate sentences.
# Paired narrative-group bootstrap of the differences, 1,000 draws, seed 2025.
import re
from transformers import AutoTokenizer
def boot_mean(values, groups, n=1000, seed=2025, mask=None):
    codes, G, cnt = group_draws(groups, n, seed); v = np.asarray(values, float); m = np.ones(len(v), bool) if mask is None else np.asarray(mask, bool)
    s = np.zeros(G); c = np.zeros(G); np.add.at(s, codes[m], v[m]); np.add.at(c, codes[m], 1.0)
    return v[m].mean(), (cnt @ s) / (cnt @ c)
EVIDENCE_IGNORE=-100; MAX_SENTENCES=16; MAX_LEN=200
# ---------- copied unchanged from notebook 07 (07_Evidence_hierarchical; runtime sha256 33200b5a...) ----------
def aux_clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+", " urltoken ", text)
    text = re.sub(r"@\w+", " usertoken ", text)
    text = re.sub(r"#(\w+)", r" hashtag_\1 ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()
SENTENCE_BOUNDARY = re.compile(r"(?<=[.!?;])\s+|[\r\n]+")
def aux_sentence_document(raw):
    raw = "" if pd.isna(raw) else str(raw)
    text = aux_clean_text(raw)
    cuts, last = [], 0
    for match in SENTENCE_BOUNDARY.finditer(raw):
        cuts.append((last, match.start()))
        last = match.end()
    cuts.append((last, len(raw)))
    sentences = []
    for raw_start, raw_end in cuts:
        start = len(aux_clean_text(raw[:raw_start]))
        end = len(aux_clean_text(raw[:raw_end]))
        while start < end and text[start].isspace():
            start += 1
        if end > start:
            sentences.append({"start": start, "end": end, "raw_start": raw_start, "raw_end": raw_end, "text": text[start:end]})
    return {"text": text, "sentences": sentences}
def aux_align_evidence(document, evidence_json, source_state):
    diag = {"quotes_total": 0, "quotes_matched": 0, "quotes_unmatched": 0, "quotes_invalid": 0, "quotes_ambiguous_occurrence": 0,
            "quotes_unknown_direction": 0, "conflicting_sentences": 0, "invalid_payload": 0, "state_quote_conflict": 0}
    try:
        quotes = json.loads(evidence_json) if isinstance(evidence_json, str) else evidence_json
        if not isinstance(quotes, list):
            raise ValueError("Evidence must be a list.")
    except (ValueError, TypeError):
        quotes, diag["invalid_payload"] = [], 1
    known_none = source_state == "UNMENTIONED" and not quotes and not diag["invalid_payload"]
    if source_state == "UNMENTIONED" and quotes:
        diag["state_quote_conflict"] = 1
    records, directions = [], [set() for _ in document["sentences"]]
    for item in quotes:
        diag["quotes_total"] += 1
        if not isinstance(item, (list, tuple)) or len(item) != 2 or not isinstance(item[0], str):
            diag["quotes_invalid"] += 1
            continue
        quote, direction = item
        cleaned = aux_clean_text(quote)
        if not cleaned:
            diag["quotes_invalid"] += 1
            continue
        value = {"FAVOURABLE": 1, "UNFAVOURABLE": 2}.get(direction, EVIDENCE_IGNORE)
        if value == EVIDENCE_IGNORE:
            diag["quotes_unknown_direction"] += 1
        occurrences, start = [], 0
        while True:
            start = document["text"].find(cleaned, start)
            if start < 0:
                break
            occurrences.append((start, start + len(cleaned)))
            start += 1
        if not occurrences:
            diag["quotes_unmatched"] += 1
            continue
        if len(occurrences) != 1:
            diag["quotes_ambiguous_occurrence"] += 1
            continue
        span = occurrences[0]
        hits = [i for i, sentence in enumerate(document["sentences"]) if span[0] < sentence["end"] and span[1] > sentence["start"]]
        if not hits:
            diag["quotes_unmatched"] += 1
            continue
        diag["quotes_matched"] += 1
        records.append({"span": list(span), "sentences": hits, "label": value})
        for i in hits:
            directions[i].add(value)
    labels = [0 if known_none else EVIDENCE_IGNORE for _ in directions]
    for i, values in enumerate(directions):
        if len(values) == 1 and EVIDENCE_IGNORE not in values and not diag["state_quote_conflict"]:
            labels[i] = next(iter(values))
        elif values:
            diag["conflicting_sentences"] += int(len(values) > 1)
    return {"labels": labels, "quotes": records, "known_none": known_none, "diagnostics": diag, "state_quote_conflict": bool(diag["state_quote_conflict"])}
def aux_sentence_ids(offsets, documents, max_sentences):   # numpy in place of torch; same assignment rule
    ids = np.full(offsets.shape[:2], -1, dtype=np.int64)
    for i, document in enumerate(documents):
        text = document["text"]; sents = document["sentences"][:max_sentences]
        for j, (start, end) in enumerate(offsets[i].tolist()):
            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1
            if end <= start:
                continue
            for k, sentence in enumerate(sents):
                if start >= sentence["start"] and end <= sentence["end"]:
                    ids[i, j] = k
                    break
    return ids
def aux_span_visible(text, span, intervals):
    start, end = span
    covered = [False] * (end - start)
    for left, right in intervals:
        for index in range(max(start, left), min(end, right)):
            covered[index - start] = True
    return all(seen or text[start + i].isspace() for i, seen in enumerate(covered))
def aux_visible_evidence(document, alignments, offsets, sentence_ids, max_sentences):
    labels = np.full((max_sentences, len(alignments)), EVIDENCE_IGNORE, dtype=np.int64)
    quote_mask = np.zeros_like(labels, dtype=bool)
    offsets = offsets.tolist() if hasattr(offsets, "tolist") else offsets
    sentence_ids = sentence_ids.tolist() if hasattr(sentence_ids, "tolist") else sentence_ids
    all_intervals = [(int(a), int(b)) for a, b in offsets if b > a]
    pooled_intervals = [(int(a), int(b)) for (a, b), sid in zip(offsets, sentence_ids) if b > a and sid >= 0]
    visible = []
    for i, sentence in enumerate(document["sentences"][:max_sentences]):
        intervals = [(max(a, sentence["start"]), min(b, sentence["end"])) for (a, b), sid in zip(offsets, sentence_ids) if sid == i and b > a]
        start = min((a for a, b in intervals), default=sentence["start"])
        end = max((b for a, b in intervals), default=start)
        complete = bool(intervals) and aux_span_visible(document["text"], (sentence["start"], sentence["end"]), intervals)
        visible.append({"sentence_index": i, "visible_start": start, "visible_end": end, "visible_text": document["text"][start:end], "n_tokens": len(intervals),
                        "has_visible_tokens": bool(intervals), "partially_truncated": bool(intervals) and not complete})
    for k, aligned in enumerate(alignments):
        if aligned["known_none"]:
            for i, sentence in enumerate(visible):
                if sentence["has_visible_tokens"]:
                    labels[i, k] = 0
        directions = [set() for _ in range(max_sentences)]
        for quote in aligned["quotes"]:
            token_visible = aux_span_visible(document["text"], quote["span"], all_intervals)
            within_limit = all(i < max_sentences for i in quote["sentences"])
            if aligned["state_quote_conflict"]:
                continue
            if not token_visible or not within_limit or not aux_span_visible(document["text"], quote["span"], pooled_intervals):
                continue
            for i in quote["sentences"]:
                directions[i].add(quote["label"])
        for i, values in enumerate(directions):
            if len(values) == 1 and EVIDENCE_IGNORE not in values:
                labels[i, k] = next(iter(values)); quote_mask[i, k] = True
    return labels, {"sentences": visible, "quote_mask": quote_mask}
# ---------- end of copied code ----------
rel=pd.read_csv(REL / f"drugscom_plus_ai_labels_{PROTOCOL_ID}.csv",usecols=["uniqueID","review","eligible_both_streams"]+[f"ai_{a}_{s}" for a in ASPECTS for s in ("state","evidence")])
split=pd.read_csv(ROOT/"00_split/data/DrugReview_split.csv"); d=rel[rel.eligible_both_streams].merge(split,on="uniqueID",validate="one_to_one"); d=d[d.split=="test"].sort_values("uniqueID").reset_index(drop=True)
docs=[aux_sentence_document(r) for r in d.review]; aligns=[[aux_align_evidence(doc,row[f"ai_{a}_evidence"],row[f"ai_{a}_state"]) for a in ASPECTS] for doc,(_,row) in zip(docs,d.iterrows())]
print("test reviews",len(d),"; aligned",flush=True)
BINS=[("1",1,1),("2--3",2,3),("4--6",4,6),("7--10",7,10),("11--16",11,16)]; rows=[]; diffs=[]
tomap={True:1.0,False:0.0,"True":1.0,"False":0.0}
rng_u=np.random.default_rng(2025)
for enc in ("albert","biobert"):
    tok=AutoTokenizer.from_pretrained(str(ROOT/RUNS["7e"]/enc/"tokenizer")); assert tok.is_fast
    texts=[doc["text"] for doc in docs]; offsets=np.zeros((len(texts),MAX_LEN,2),dtype=np.int64)
    for i in range(0,len(texts),2000):
        e=tok(texts[i:i+2000],truncation=True,padding="max_length",max_length=MAX_LEN,return_offsets_mapping=True); offsets[i:i+2000]=np.array(e["offset_mapping"])
    sids=aux_sentence_ids(offsets,docs,MAX_SENTENCES); print(enc,"tokenized",flush=True)
    e1=pd.read_csv(ROOT/RUNS["7e"]/enc/f"{enc}_test_predictions.csv").sort_values("uniqueID").reset_index(drop=True); e0=pd.read_csv(ROOT/RUNS["7c"]/enc/f"{enc}_test_predictions.csv").sort_values("uniqueID").reset_index(drop=True)
    assert (e1.uniqueID.values==d.uniqueID.values).all() and (e0.uniqueID.values==d.uniqueID.values).all()
    rec={a:{"elig":np.zeros(len(d),bool),"ncand":np.zeros(len(d),int),"uniform":np.zeros(len(d)),"first":np.zeros(len(d)),"longest":np.zeros(len(d)),"random":np.zeros(len(d)),"top_in_vis":np.zeros(len(d),bool)} for a in ASPECTS}
    for i,(doc,al) in enumerate(zip(docs,aligns)):
        labels,item=aux_visible_evidence(doc,al,offsets[i],sids[i],MAX_SENTENCES); qm=item["quote_mask"]; vis=item["sentences"]
        cand=[s["sentence_index"] for s in vis if s["has_visible_tokens"]]; ntok={s["sentence_index"]:s["n_tokens"] for s in vis}; vtext={s["visible_text"] for s in vis if s["has_visible_tokens"]}
        for k,a in enumerate(ASPECTS):
            hits=[j for j in cand if qm[j,k]]; r=rec[a]; r["elig"][i]=len(hits)>0; r["ncand"][i]=len(cand)
            if hits:
                r["uniform"][i]=len(hits)/len(cand); r["first"][i]=float(cand[0] in hits); r["longest"][i]=float(max(cand,key=lambda j:(ntok[j],-j)) in hits); r["random"][i]=float(rng_u.choice(cand) in hits)
                r["top_in_vis"][i]=str(e1.loc[i,f"{a}_top_visible_sentence"]) in vtext
    groups=d.narrative_group.values
    for a in ASPECTS:
        r=rec[a]; v1=e1[f"{a}_has_visible_selected_quote"].astype(bool).values; v0=e0[f"{a}_has_visible_selected_quote"].astype(bool).values
        mism=int((r["elig"]!=v1).sum()); print(enc,a,"eligibility mismatches vs exported eta=1 column:",mism,"| vs eta=0:",int((r["elig"]!=v0).sum()),"| top sentence text found among re-derived visible sentences:",round(float(r["top_in_vis"][r["elig"]].mean()),4),flush=True)
        h1=e1[f"{a}_top1_selected_quote_hit"].map(tomap).values; h0=e0[f"{a}_top1_selected_quote_hit"].map(tomap).values; el=r["elig"]&v1
        for name,lo,hi in BINS+[("all",1,10**6)]:
            m=el&(r["ncand"]>=lo)&(r["ncand"]<=hi)
            if m.sum()==0: continue
            rows.append({"encoder":enc,"aspect":a,"candidates":name,"n":int(m.sum()),"eta1":float(np.nanmean(h1[m])),"eta0":float(np.nanmean(h0[m])),"uniform":float(r["uniform"][m].mean()),"random_draw":float(r["random"][m].mean()),"first":float(r["first"][m].mean()),"longest":float(r["longest"][m].mean()),"mismatch_eligibility":mism})
        for name,x,y in (("eta1 - uniform",h1,r["uniform"]),("eta1 - first",h1,r["first"]),("eta1 - longest",h1,r["longest"]),("eta0 - uniform",h0,r["uniform"]),("eta0 - first",h0,r["first"]),("eta0 - longest",h0,r["longest"]),("eta1 - eta0",h1,h0)):
            m2=el&(r["ncand"]>=2); pt,bt=boot_mean(np.nan_to_num(x)-np.nan_to_num(y),groups,1000,2025,mask=m2); lo_,hi_=ci(bt)
            diffs.append({"encoder":enc,"aspect":a,"contrast":name,"n_reviews_2plus_candidates":int(m2.sum()),"delta":pt,"lo":lo_,"hi":hi_})
save(pd.DataFrame(rows), "localization_baselines.csv", index=False); save(pd.DataFrame(diffs), "localization_baselines_diffs.csv", index=False)
pd.set_option("display.width",250); print(pd.DataFrame(rows).round(3).to_string(index=False)); print(pd.DataFrame(diffs).round(3).to_string(index=False))

test reviews 20298 ; aligned
albert tokenized
albert efficacy eligibility mismatches vs exported eta=1 column: 0 | vs eta=0: 0 | top sentence text found among re-derived visible sentences: 1.0
albert safety eligibility mismatches vs exported eta=1 column: 0 | vs eta=0: 0 | top sentence text found among re-derived visible sentences: 1.0
albert burden eligibility mismatches vs exported eta=1 column: 0 | vs eta=0: 0 | top sentence text found among re-derived visible sentences: 1.0
albert cost eligibility mismatches vs exported eta=1 column: 0 | vs eta=0: 0 | top sentence text found among re-derived visible sentences: 1.0
biobert tokenized
biobert efficacy eligibility mismatches vs exported eta=1 column: 0 | vs eta=0: 0 | top sentence text found among re-derived visible sentences: 1.0
biobert safety eligibility mismatches vs exported eta=1 column: 0 | vs eta=0: 0 | top sentence text found among re-derived visible sentences: 1.0
biobert burden eligibility mismatches vs exported eta=1 column